# Tariff shock projections

Set the scenario family, cases and output detail below, then use **Run All**. Model 1 runs groups 1 to 5 under low, central and high calibration. Each combination writes four CSV tables to the configured run's shock_tfp folder. Repeating a combination replaces its tables.

In [ ]:
NEMESIS_SCENARIO = "TariffsAndSub"  # Family: "TariffsAndSub" or "Tariff"
CASES = ["C"]                      # A: imports only; B: domestic replacement; C: NEMESIS production
SIMPLE = True                      # False also exports supplying-bloc detail

In [ ]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import sys
project_root = next(parent for parent in (Path.cwd(), *Path.cwd().parents)
                    if (parent / "code" / "project_settings.py").is_file())
sys.path.insert(0, str(project_root / "code"))
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, YEARS, RESULTS_DIR, SRC_DIR, SRC_DIRS, MATRICES_DIR, INTERMEDIATE_DIR

CONTENT = "dir"        # "dir" : rd_dir_CI | "tot" : rd_tot_CI
YEAR = 2020 # YEARS[0]       # year of the base table (the same for all the years of YEARS_CALC)
YEARS_CALC = list(range(2025, 2051, 5))                # years computed (the omega file has one column per 5 years): the shock of year t is the level gap vs the baseline of year t (nothing to cumulate)
DISPLAY_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]   # years displayed in the table (among YEARS_CALC); YEAR = base year, variation 0
MODELS = [1]          # [1]: one elasticity for the total domestic and foreign R&D stock

EU27 = ["AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT",
        "RO", "SK", "SI", "ES", "SE"]
BLOCKS = {
    "USA": ["US"],
    "EU27": EU27,
    "RestOECD": ["AU", "CA", "JP", "KR", "MX", "NZ", "NO", "CH", "UK"],
    "China": ["CN"],
    "RestBRICS": ["BR", "RU", "IN", "ZA"],
    "RestAsia": ["BD", "KH", "ID", "LA", "MY", "MM", "NP", "PK", "PH", "SG", "TW", "TJ", "TL", "TH", "VN"],
}
REGION_TO_BLOCK = {region: block for block, regions in BLOCKS.items() for region in regions}
DEFAULT_BLOCK = "RoW"

MANUFACTURING = ["C13_C15", "C16_C18", "C20+C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31_C33"]
NONMANUF = "NONMANUF"
AGGREGATES = ["C", "H"]    # sector groups (prefix of the sector codes) also computed as one aggregate sector, as add_sector() of the 2.0

# ---- final file in IAMC format: Model | Scenario | Region | Variable | Unit | one column per year -----------------------------------
# Combined export: Region is the importer; Variable is the sector total or "sector | supplying bloc".
IAMC_MODEL = "NEMESIS-World v1.1"                                     # "Model" column
IAMC_SCENARIO = "model{model}_case{scenario}"  # "Scenario" column: {model} = 2 / 3 / 4 (model of blocks), {scenario} = A / B
IAMC_VARIABLE = "{sector} | {origin}"                        # "Variable" column: {sector} = sector whose unit cost changes ("C" = all manufacturing), {origin} = bloc supplying the inputs
# English NACE Rev. 2 labels and project aggregates, applied only to the final export.
# H includes H53+J in the current project aggregation.
SECTOR_NAMES = {'A': 'Agriculture, forestry and fishing',
 'B': 'Mining and quarrying',
 'C': 'Manufacturing',
 'C10_C12': 'Food, beverages and tobacco',
 'C13_C15': 'Textiles, wearing apparel and leather products',
 'C16_C18': 'Wood, paper and printing products',
 'C19': 'Coke and refined petroleum products',
 'C20+C21': 'Chemicals and pharmaceuticals',
 'C22': 'Rubber and plastic products',
 'C23': 'Other non-metallic mineral products',
 'C24': 'Basic metals',
 'C25': 'Fabricated metal products',
 'C26': 'Computer, electronic and optical products',
 'C27': 'Electrical equipment',
 'C28': 'Machinery and equipment n.e.c.',
 'C29': 'Motor vehicles, trailers and semi-trailers',
 'C30': 'Other transport equipment',
 'C31_C33': 'Furniture, other manufacturing, machinery repair and installation',
 'D': 'Electricity, gas, steam and air conditioning supply',
 'E': 'Water supply, sewerage, waste management and remediation',
 'F': 'Construction',
 'G': 'Wholesale and retail trade, motor vehicle repair',
 'H': 'Transportation, storage, information and communication services',
 'H49': 'Land transport and transport via pipelines',
 'H50': 'Water transport',
 'H51': 'Air transport',
 'H52': 'Warehousing and support activities for transportation',
 'H53+J': 'Postal, courier, information and communication services',
 'I': 'Accommodation and food service activities',
 'K': 'Financial and insurance activities',
 'L': 'Real estate activities',
 'M+N': 'Professional, scientific, technical, administrative and support services',
 'O+U': 'Public administration, defence, social security and extraterritorial organisations',
 'P': 'Education',
 'Q': 'Human health and social work activities',
 'R+S+T': 'Arts, entertainment, recreation, other services and household activities'}

IAMC_YEARS = YEARS_CALC                                       # years written, one column each (among YEARS_CALC)
IAMC_CSV = dict(sep=";", decimal=",")                         # csv format (";" and "," = French Excel opens it directly)
IAMC_XLSX = False                                              # also write the same table as .xlsx

In [ ]:
def read_matrix(path):
    return pd.read_csv(path, sep=";", header=[0, 1, 2], index_col=[0, 1, 2])


def flat(m):
    m = m.copy()
    m.index, m.columns = m.index.droplevel(1), m.columns.droplevel(1)
    m.index.names = m.columns.names = ["Region", "Sector"]
    return m


MODEL_BLOCKS = {
    1: {"T": (None, None)},
    2: {"D": (None, True), "F": (None, False)},
    3: {"I": (True, None), "X": (False, None)},
    4: {"same_same": (True, True), "same_diff": (True, False), "diff_same": (False, True), "diff_diff": (False, False)},
}


def block_matrices(m, region_to_block=REGION_TO_BLOCK, default_block=DEFAULT_BLOCK):
    reg_i, reg_j = m.index.get_level_values("Region"), m.columns.get_level_values("Region")
    foreign = np.asarray(reg_i)[:, None] != np.asarray(reg_j)[None, :]
    rows = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_i], m.index.get_level_values("Sector")], names=["Bloc", "Sector"])
    cols = pd.MultiIndex.from_arrays([[region_to_block.get(r, default_block) for r in reg_j], m.columns.get_level_values("Sector")], names=["Bloc", "Sector"])
    out = []
    for mask in (~foreign, foreign):
        a = pd.DataFrame(m.to_numpy() * mask, index=rows, columns=cols)
        out.append(a.groupby(level=["Bloc", "Sector"]).sum().T.groupby(level=["Bloc", "Sector"]).sum().T)
    return out


def omega_blocks(index, columns, omega, manufacturing=None):      # manufacturing=None: every sector of the omega file is shocked (a list restricts the shock to these sectors)
    b_i, sec_i = np.asarray(index.get_level_values("Bloc")), np.asarray(index.get_level_values("Sector"))
    b_j = np.asarray(columns.get_level_values("Bloc"))
    om = np.ones((len(index), len(columns)))
    for r in omega.sort_values("sector", key=lambda s: s != "ALL").itertuples():
        rows = (b_i == r.origin) & ((sec_i == r.sector) | (r.sector == "ALL"))
        om[np.ix_(rows, b_j == r.dest)] = 1 + r.delta_pct / 100
    return om if manufacturing is None else np.where(np.isin(sec_i, manufacturing)[:, None], om, 1.0)


def shock_bilateral(cd, cf, zd, zf, omega, beta, model, aggregates=AGGREGATES, manufacturing=None, pi=None):
    """Same cost shock as shock_blocks + add_sector, but BY BLOC OF ORIGIN of the inputs: dlnCost of the sector x of the bloc y due to what it buys
    from the bloc z. One row per (scenario, bloc y, sector x, origin z).
    - The change of the stock S_k of a buyer is a sum over the supplying blocs z: dS_k(z) = sum over the suppliers i of the bloc z of
      c_ij * (omega_ij - 1) (same masks k as shock_blocks). Scenario B adds the substitution gain g (domestic production fills the gap), attributed
      to the origin z = bloc of the buyer itself.
    - Scenario C (pi given: columns bloc, sector, delta_pct = change in % of the NEMESIS production of the bloc and sector vs the reference): the domestic
      content bought by the buyer from a supplier sector changes like the production of that sector in the bloc of the buyer, gain = c_dom * delta_pct / 100
      (not below -c_dom). The foreign flows are shocked as in A; no substitution is assumed, the volumes z are not needed.
    - dlnS_k = log((S0 + sum_z dS_k(z)) / S0) is not additive: it is split between the origins in proportion to their share of the stock change,
      dlnS_k(z) = dlnS_k * dS_k(z) / sum_z dS_k(z), so that the sum over z gives back the dlnS_k of shock_blocks exactly.
    - dlnCost(z) = sum_k beta_k * dlnS_k(z): the sum over z is the dlnCost of shock_blocks + add_sector.
    - aggregates (e.g. "C", "H"): the stocks and the changes of the sectors of the group are summed first (as add_sector), then dlnS and the split."""
    idx, cols = cd.index, cd.columns
    bloc_i, sec_i = np.asarray(idx.get_level_values("Bloc")), np.asarray(idx.get_level_values("Sector"))
    bloc_j, sec_j = np.asarray(cols.get_level_values("Bloc")), np.asarray(cols.get_level_values("Sector"))
    origins = np.array(list(dict.fromkeys(bloc_i)))
    n_o, n_d = len(origins), len(cols)
    members = (origins[:, None] == bloc_i[None, :]).astype(float)        # (origin, supplier): the supplier belongs to the bloc
    own = origins[:, None] == bloc_j[None, :]                              # (origin, buyer): the origin is the bloc of the buyer
    same_sec = sec_i[:, None] == sec_j[None, :]
    om = omega_blocks(idx, cols, omega, manufacturing)
    cdv, cfv = cd.to_numpy(), cf.to_numpy()
    types = MODEL_BLOCKS[model]

    s0, shock = {}, {}
    for k, (sec, cty) in types.items():
        ms = np.ones_like(same_sec) if sec is None else same_sec == sec
        d = (cdv * ms).sum(0) if cty is not False else 0.0                  # domestic part (never shocked)
        f0 = (cfv * ms).sum(0) if cty is not True else 0.0
        s0[k] = d + f0
        shock[k] = members @ (cfv * (om - 1.0) * ms) if cty is not True else np.zeros((n_o, n_d))   # change of S_k by origin bloc

    none = np.zeros(n_d)
    cases = [("A", none, none)]
    by_sector = lambda a: pd.DataFrame(a, index=idx, columns=cols).groupby(level="Sector").sum()
    c_dom = by_sector(cdv)             # domestic R&D content bought from each supplier sector (rows) by each buyer (columns)

    def same_other(gain):              # gain (supplier sector x buyer) -> gain in the sector of the buyer itself / in the other sectors
        g_same = gain.to_numpy()[gain.index.get_indexer(sec_j), np.arange(len(sec_j))]
        return g_same, gain.sum().to_numpy() - g_same

    if zd is not None:                 # scenario B: needs the volumes z
        z_dom = by_sector(zd.to_numpy())
        lost = by_sector((1 - om) * zf.to_numpy())
        rho_d = (c_dom / z_dom.where(z_dom > 0)).fillna(0.0)
        gain = np.maximum(rho_d * lost, -c_dom)
        cases.append(("B", *same_other(gain)))
    if pi is not None:                 # scenario C: the domestic content follows the NEMESIS production of the bloc of the buyer
        change = pi.pivot(index="sector", columns="bloc", values="delta_pct").div(100.0).reindex(index=c_dom.index, columns=bloc_j).fillna(0.0)
        change.columns = c_dom.columns
        gain = np.maximum(c_dom * change, -c_dom)
        cases.append(("C", *same_other(gain)))

    def split(S0, dS):
        """S0: stock before; dS: change of the stock by origin (array over the origins). Returns dlnS by origin."""
        tot = dS.sum(0)
        with np.errstate(all="ignore"):
            dln = np.where(S0 > 0, np.log((S0 + tot) / S0), 0.0)
            share = np.where(tot != 0, dS / np.where(tot != 0, tot, 1.0), 0.0)
        return dln * share

    out = []
    for scenario, g_same, g_diff in cases:
        dS = {}
        for k, (sec, cty) in types.items():
            g = 0.0 if cty is False else (g_same + g_diff if sec is None else g_same if sec else g_diff)
            dS[k] = shock[k] + own * g                                       # the gain g is attributed to the bloc of the buyer
        res = pd.DataFrame({"bloc": np.tile(bloc_j, n_o), "sector": np.tile(sec_j, n_o), "origin": np.repeat(origins, n_d)})
        for k in types:
            res[f"dlnS_{k}"] = split(s0[k], dS[k]).ravel()
        parts = [res]
        for a in aggregates:
            in_a = np.array([s.startswith(a) for s in sec_j])
            for b in dict.fromkeys(bloc_j):
                sel = in_a & (bloc_j == b)
                if not sel.any():
                    continue
                row = pd.DataFrame({"bloc": b, "sector": a, "origin": origins})
                for k in types:
                    S0 = s0[k][sel].sum()
                    dS_a = (dS[k][:, sel] * (s0[k][sel] > 0)).sum(1)          # stocks without a baseline stay at 0, as add_sector
                    row[f"dlnS_{k}"] = split(S0, dS_a)
                parts.append(row)
        out.append(pd.concat(parts, ignore_index=True).assign(scenario=scenario))
    res = pd.concat(out, ignore_index=True)
    if beta is not None:
        res = res.merge(beta, on="sector", how="left")
        res["dlnCost"] = sum(res[f"beta_{k}"] * res[f"dlnS_{k}"] for k in types)
    return res

In [ ]:
if not CASES or not set(CASES) <= {"A", "B", "C"}:
    raise ValueError(f"CASES must be a list made of 'A', 'B' and/or 'C', got {CASES!r}")
if not MODELS or not set(MODELS) <= set(MODEL_BLOCKS):
    raise ValueError(f"MODELS must be a list made of {list(MODEL_BLOCKS)}, got {MODELS!r}")


if BASE not in ("EXIOBASE", "FIGARO", "OECD_ICIO", "GTAP"):
    raise ValueError(f"BASE={BASE!r} : le choc utilise les flux Z d'une base MRIO (EXIOBASE, FIGARO, OECD_ICIO ou GTAP)")


Z_REF_YEAR = 2010   # year whose z_CI is used to rebuild the volumes of the years without z_CI (fixed technical coefficients)


def mpath(name, year, suffix="_agg"):
    return Path(SRC_DIR) / BASE / f"output_loop_{BASE}_{year}" / f"{name}_{BASE}_{year}{suffix}.csv"


prod = pd.read_csv(Path(SRC_DIR) / BASE / f"output_loop_{BASE}" / f"production_{BASE}_agg.csv", sep=";").set_index(["Region", "Sector"])
_cache = {}


def load_z(c, year):
    if mpath("z_CI", year).exists():
        return flat(read_matrix(mpath("z_CI", year))).reindex(index=c.index, columns=c.columns).fillna(0.0), "z_CI exact"
    if "A" not in _cache:
        for d in (SRC_DIR, MATRICES_DIR + "SRDV_c_s/", MATRICES_DIR + "SRDQ_c_s/"):   # z_CI of the reference year, in any run folder
            p = Path(d) / BASE / f"output_loop_{BASE}_{Z_REF_YEAR}" / f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv"
            if p.exists():
                z_ref = flat(read_matrix(p)).reindex(index=c.index, columns=c.columns).fillna(0.0)
                x_ref = prod[str(Z_REF_YEAR)].reindex(c.index).fillna(0.0).to_numpy()
                _cache["A"] = z_ref.to_numpy() / np.where(x_ref > 0, x_ref, 1.0)[None, :]
                break
        else:
            raise FileNotFoundError(f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv introuvable : lancer know_calc_loop4.0 pour l'année {Z_REF_YEAR}")
    x = prod[str(year)].reindex(c.index).fillna(0.0).to_numpy()
    z_prox = _cache["A"] * x[None, :]                                    # volumes with the technical coefficients of Z_REF_YEAR
    zs, cs = z_prox.sum(axis=1), c.to_numpy().sum(axis=1)
    rho = np.divide(cs, zs, out=np.zeros(len(zs)), where=zs > 0)         # R&D of the supplier per euro sold this year
    z = np.where(rho[:, None] > 0, np.divide(c.to_numpy(), rho[:, None], out=np.zeros(c.shape), where=rho[:, None] > 0), z_prox)   # z = c / rho
    return pd.DataFrame(z, index=c.index, columns=c.columns), f"z reconstruit (c/rho, A {Z_REF_YEAR})"


def find(name):
    for d in SRC_DIRS:
        p = Path(d) / BASE / f"output_loop_{BASE}_{YEAR}" / f"{name}_{BASE}_{YEAR}_agg.csv"
        if p.exists():
            return p
    raise FileNotFoundError(f"{name}_{BASE}_{YEAR}_agg.csv introuvable dans {SRC_DIRS} : lancer know_calc_loop puis aggregate_to_common_mapping")


inputs = Path(PATH_WRK) / "shock_tfp" / "inputs"
def read_omega(path):
    """Omega file: one column per year (origin, dest, sector, 2025, 2030, ...) as made by make_omega.py, or the long format (delta_pct column, year column optional)."""
    om = pd.read_csv(path)
    if "delta_pct" in om.columns:
        return om
    years = [c for c in om.columns if str(c).isdigit()]
    om = om.melt(id_vars=["origin", "dest", "sector"], value_vars=years, var_name="year", value_name="delta_pct")
    om["year"] = om["year"].astype(int)
    return om.dropna(subset=["delta_pct"])


prefix = {"dir": "rd_dir_CI", "tot": "rd_tot_CI"}[CONTENT]
c0 = flat(read_matrix(find(prefix)))
if "B" in CASES:                              # only case B needs the volumes z (A does not)
    z0, z_src = load_z(c0, YEAR)
else:
    z0, z_src = None, "cas A : pas de volumes z"
print(f"volumes {YEAR} : {z_src}")

cd, cf = block_matrices(c0)                       # domestic / foreign content, aggregated by (block, sector) BEFORE the shock
zd, zf = (None, None) if z0 is None else [m.reindex(index=cd.index, columns=cd.columns).fillna(0.0) for m in block_matrices(z0)]
print(f"{cd.shape[0]} (bloc, secteur) fournisseurs x {cd.shape[1]} (bloc, secteur) acheteurs")

out = Path(RESULTS_DIR) / "shock_tfp"
out.mkdir(parents=True, exist_ok=True)


# ---- every output table is written as CSV and, next to it, as .xlsx (IAMC_XLSX) with a second sheet "Description": scenario, case, model, definitions
MODEL_TEXT = {
    1: "Model 1: one elasticity (beta_T) applies to the total domestic and foreign R&D stock.",
    2: "Model 2: the unit cost depends on 2 R&D content stocks, bought from domestic suppliers (D) and from foreign suppliers (F).",
    3: "Model 3: the unit cost depends on 2 R&D content stocks, bought from the same sector (I) and from the other sectors (X).",
    4: "Model 4: the unit cost depends on 4 R&D content stocks: same sector / other sectors x domestic / foreign suppliers.",
}
BETA_TEXT = {
    "_low": "Model 1 low calibration",
    "_central": "Model 1 central calibration",
    "_high": "Model 1 high calibration",
    "": "betas of the files without suffix",
    "_bounded": "betas bounded to [-0.25, 0]",
    "_trend": "betas estimated with year fixed effects (trend), not bounded",
    "_trend_bounded": "betas estimated with year fixed effects (trend), bounded to [-0.25, 0]",
}


def beta_file(model):
    return f"beta{BETA_SUFFIX}.csv" if model == 2 else f"beta_model{model}{BETA_SUFFIX}.csv"


CASE_TEXT = {
    "A": "Case A: no substitution, only the foreign R&D content bought by the sector changes (omega applied to the foreign flows).",
    "B": "Case B: substitution, domestic production fills the gap left by the foreign flows (gain attributed to the bloc of the buyer itself; needs the volumes z).",
    "C": "Case C: the foreign flows are shocked as in case A, and the domestic R&D content bought by a sector changes like the NEMESIS production of the supplier sector in the bloc of the buyer "
         "(scenario / reference), no assumed substitution (gain attributed to the bloc of the buyer itself).",
}


def describe(what):
    """Rows of the sheet "Description": what the table is, then the scenario, case(s), model(s) and definitions used to compute it."""
    from datetime import datetime
    rows = [
        ("Table", what),
        ("NEMESIS scenario", f"{scenario_name} (omega file: {OMEGA_FILE})"),
        ("Omega (import shock)", "delta_pct = (bilateral exports of the scenario / bilateral exports of the reference scenario - 1) * 100, NEMESIS variables "
                                 "'Trade|Export <bloc A> to <bloc B>|<sector>', one value per 5 years, all the sectors. Applied to the foreign R&D content bought "
                                 "by the sectors of bloc B from bloc A."),
    ]
    if "C" in CASES:
        rows.append(("Production (case C)", f"delta_pct = (production of the scenario / production of the reference scenario - 1) * 100, NEMESIS variables 'Trade|Production|<sector>' "
                                            f"summed over the regions of each bloc, one value per 5 years (file {PROD_FILE})."))
    rows += [(f"Case {c}", CASE_TEXT[c]) for c in CASES]
    rows += [(f"Model {m}", MODEL_TEXT[m]) for m in MODELS]
    rows.append(("Betas", f"{BETA_TEXT.get(BETA_SUFFIX, 'suffix ' + repr(BETA_SUFFIX))}; files: " + ", ".join(beta_file(m) for m in MODELS) + "."))
    rows += [
        ("Years", f"base year {YEAR}; years computed {YEARS_CALC}; years displayed {DISPLAY_YEARS}. Each year is a level gap vs the baseline of the same year (nothing is cumulated)."),
        ("R&D content", f"{BASE} matrix, {'rd_dir_CI (direct)' if CONTENT == 'dir' else 'rd_tot_CI (total)'}, base year {YEAR}."),
        ("Blocs", "; ".join(f"{b} = {', '.join(r)}" for b, r in BLOCKS.items()) + f"; {DEFAULT_BLOCK} = all the other countries."),
        ("Unit and definition", "%: change in the unit cost of the sector of the importing bloc (Region), relative to the baseline. Sector total = 100 * (exp(sum over the supplying "
                                "blocs of dlnCost) - 1); 'sector | origin' = same formula for one supplying bloc only. Bilateral rows add up in logs, not in percentages."),
        ("Non-manufacturing", f"Aggregate of the sectors that are not manufacturing (not C*, not the aggregate H): average of the log cost variations of these sectors weighted by their "
                              f"production in the bloc ({BASE} production file, base year {YEAR}, countries of the bloc summed); weights renormalised on the sectors that have a cost variation."),
        ("All sectors", f"Aggregate of the manufacturing (aggregate C, computed from the R&D stocks) and of the non-manufacturing (aggregate above): average of their log cost variations weighted by their "
                        f"production in the bloc ({BASE} production file, base year {YEAR}: sum of the manufacturing sectors, sum of the non-manufacturing sectors); only where both are available."),
        ("Totals only" if SIMPLE else "Detail", "Only the sector totals are exported (SIMPLE = True)." if SIMPLE else "Sector totals and the detail by supplying bloc."),
        ("Case column", "model<model>_case<case>, e.g. " + IAMC_SCENARIO.format(model=MODELS[0], scenario=CASES[0]) + "."),
        ("Generated", datetime.now().strftime("%Y-%m-%d %H:%M")),
    ]
    return pd.DataFrame(rows, columns=["Item", "Description"])


def write_table(df, path, what, **csv_kwargs):
    """CSV as before + (IAMC_XLSX) the same table in an .xlsx with the sheets "Data" and "Description"."""
    df.to_csv(path, index=False, **csv_kwargs)
    if IAMC_XLSX:
        from openpyxl.styles import Alignment
        with pd.ExcelWriter(Path(path).with_suffix(".xlsx"), engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="Data", index=False)
            describe(what).to_excel(writer, sheet_name="Description", index=False)
            ws = writer.sheets["Description"]
            ws.column_dimensions["A"].width, ws.column_dimensions["B"].width = 24, 120
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.alignment = Alignment(wrap_text=True, vertical="top")

def load_scenario_inputs(OMEGA_FILE, PROD_FILE):
    omega_all = read_omega(inputs / OMEGA_FILE)                                       # ω of all the years (year column optional)

    prod_all = None
    if "C" in CASES:                                                                  # case C: change (%) of the NEMESIS production by (bloc, sector, year), made by make_prod.py
        if not (inputs / PROD_FILE).exists():
            raise FileNotFoundError(f"{inputs / PROD_FILE} introuvable : le créer avec make_prod.py (cas C)")
        _pr = pd.read_csv(inputs / PROD_FILE)
        prod_all = _pr.melt(id_vars=["bloc", "sector"], value_vars=[c for c in _pr.columns if str(c).isdigit()], var_name="year", value_name="delta_pct")
        prod_all["year"] = prod_all["year"].astype(int)


    return omega_all, prod_all

In [ ]:
def omega_of(year):
    if "year" not in omega_all.columns:
        return omega_all
    om = omega_all[omega_all["year"] == year].drop(columns="year")
    if om.empty:
        raise ValueError(f"{OMEGA_FILE} : aucune ligne pour l'année {year}")
    return om


def pi_of(year):
    if prod_all is None:
        return None
    p = prod_all[prod_all["year"] == year].drop(columns="year")
    if p.empty:
        raise ValueError(f"{PROD_FILE} : aucune ligne pour l'année {year}")
    return p


def calculate_results():
    results = {}
    for model in MODELS:
        f_beta = inputs / beta_file(model)
        if model == 1 and not f_beta.exists():
            raise FileNotFoundError(f"{f_beta} not found. Model 1 uses _low, _central or _high.")
        if BETA_SUFFIX and not f_beta.exists():
            raise FileNotFoundError(f"{f_beta} introuvable : le creer avec make_beta.py --suffix {BETA_SUFFIX}")
        beta_m = pd.read_csv(f_beta) if f_beta.exists() else None
        cache, frames = {}, []
        for year in YEARS_CALC:
            omega = omega_of(year)
            pi = pi_of(year)
            key = omega.to_csv(index=False) + ("" if pi is None else pi.to_csv(index=False))
            if key not in cache:
                res_m = shock_bilateral(cd, cf, zd, zf, omega, beta_m, model, pi=pi)
                cache[key] = res_m[res_m["scenario"].isin(CASES)]
            frames.append(cache[key].assign(year=year))
        results[model] = pd.concat(frames, ignore_index=True)
        write_table(results[model], out / f"results_model{model}_blocs_bilateral_{BASE}_{CONTENT}_horizon_{scenario_name}{BETA_SUFFIX}.csv",
                    "Detailed results: one row per (case, importing bloc, sector, supplying bloc, year), log changes of the R&D content stocks (dlnS) and of the unit cost (dlnCost).")
        print(f"modèle ({model}) : {len(YEARS_CALC)} années, {len(results[model])} lignes (bloc, secteur, bloc d'origine) écrites dans {out}")
    return results

In [ ]:
def export_bilateral(results):
    # Table of the displayed years: one row per (scenario, sector, bloc, origin bloc), one column per year. Change in unit cost of the sector of the bloc
    # due to what it buys from the origin bloc, relative to the baseline (the base year does not move: 0)
    tables = {}
    for model in MODELS:
        if "dlnCost" not in results[model].columns:
            continue
        wide = results[model].pivot_table(index=["scenario", "sector", "bloc", "origin"], columns="year", values="dlnCost").reindex(columns=DISPLAY_YEARS)
        wide.insert(0, YEAR, 0.0)
        wide = (np.exp(wide) - 1) * 100
        wide.columns.name = None
        tables[model] = wide.round(4).reset_index()
        path = out / f"bilateral_cost_change_{BASE}_{CONTENT}_model{model}_base{YEAR}_{scenario_name}{BETA_SUFFIX}.csv"
        write_table(tables[model], path, "Change in unit cost (%) by (case, sector, importing bloc, supplying bloc), one column per displayed year (the base year is 0).")
        print(f"modèle ({model}) : tableau de {len(tables[model])} lignes écrit dans {path}")

In [ ]:
# One final table: sector totals and the detail by supplying bloc.
# Both use the SAME bilateral dlnCost values, computed in the preceding cells.
# Sector total (%) = 100 * (exp(sum over origins of dlnCost) - 1).
# Bilateral row (%) = 100 * (exp(dlnCost for this origin) - 1), as in the earlier export.
# The bilateral rows add up in LOGS, not in percentages. Do not sum the exported percentages.
# Each year is a level difference from its own baseline; nothing is cumulated over years.

IAMC_UNIT = "%"

# ---- Non-manufacturing aggregate: the sectors that are not manufacturing (not C*, not the aggregate H) are averaged with their PRODUCTION in the bloc as weight.
# Not computed like "C" (sum of the R&D stocks, beta of the aggregate): there is no beta for this group and no weight inside it, so the cost variations
# of the sectors (each with its own beta) are averaged. The average is made on dlnCost, which is additive over the origins: the sum over the origins of the
# weighted averages is the weighted average of the sector totals.
NONMANUF_NAME = "Non-manufacturing"
SECTOR_NAMES[NONMANUF] = NONMANUF_NAME
ALLSECT, ALLSECT_NAME = "ALLSECT", "All sectors"
SECTOR_NAMES[ALLSECT] = ALLSECT_NAME
is_manuf = lambda s: s.startswith("C") and s not in AGGREGATES
is_nonmanuf = lambda s: not s.startswith("C") and s not in AGGREGATES and s not in (NONMANUF, ALLSECT)


def production_weights(keep, year=YEAR):
    """Production of each sector (selected by keep) in each bloc (production file of BASE, base year, countries of the bloc summed)."""
    p = prod[str(year)].reset_index()
    p["bloc"] = p["Region"].map(REGION_TO_BLOCK).fillna(DEFAULT_BLOCK)
    w = p.groupby(["bloc", "Sector"])[str(year)].sum().rename("w").reset_index().rename(columns={"Sector": "sector"})
    return w[(w["w"] > 0) & w["sector"].map(keep)]


W_NONMANUF = production_weights(is_nonmanuf)
W_MANUF = production_weights(is_manuf)
# All sectors = aggregate "C" + aggregate NONMANUF, each weighted by the total production of its sectors in the bloc
W_ALLSECT = pd.concat([W_MANUF.groupby("bloc", as_index=False)["w"].sum().assign(sector="C"),
                       W_NONMANUF.groupby("bloc", as_index=False)["w"].sum().assign(sector=NONMANUF)], ignore_index=True)
print(f"{ALLSECT_NAME}: C + {NONMANUF_NAME}, weights = total production of the manufacturing / of the non-manufacturing (blocs {', '.join(sorted(W_ALLSECT['bloc'].unique()))})")
print(f"{NONMANUF_NAME}: weights = production {BASE} {YEAR}, {W_NONMANUF['sector'].nunique()} sectors ({', '.join(sorted(W_NONMANUF['sector'].unique()))}); "
      f"blocs {', '.join(sorted(W_NONMANUF['bloc'].unique()))}")


def add_nonmanuf(long):
    """Adds the rows of the aggregate NONMANUF (case, bloc, origin, year): production-weighted average of the dlnCost of the non-manufacturing sectors,
    weights renormalised on the sectors that have a dlnCost."""
    nm = long.merge(W_NONMANUF, on=["bloc", "sector"], how="inner")
    nm = nm.assign(wd=nm["w"] * nm["dlnCost"])
    agg = nm.groupby(["scenario", "bloc", "origin", "year"], as_index=False).agg(wd=("wd", "sum"), w=("w", "sum"))
    agg["dlnCost"], agg["sector"] = agg["wd"] / agg["w"], NONMANUF
    return pd.concat([long, agg[list(long.columns)]], ignore_index=True)


def add_allsect(long):
    """Adds the rows of the aggregate ALLSECT (to be called after add_nonmanuf): production-weighted average of the dlnCost of the aggregates C and NONMANUF
    (weights = W_ALLSECT). Only where both aggregates have a dlnCost (otherwise it would be the other one alone)."""
    al = long[long["sector"].isin(["C", NONMANUF])].merge(W_ALLSECT, on=["bloc", "sector"], how="inner")
    al = al.assign(wd=al["w"] * al["dlnCost"])
    agg = al.groupby(["scenario", "bloc", "origin", "year"], as_index=False).agg(wd=("wd", "sum"), w=("w", "sum"), n=("sector", "nunique"))
    agg = agg[agg["n"] == 2]
    agg["dlnCost"], agg["sector"] = agg["wd"] / agg["w"], ALLSECT
    return pd.concat([long, agg[list(long.columns)]], ignore_index=True)


def to_iamc(simple, results, years=DISPLAY_YEARS):
    """Export sector totals and origin rows from a single common log-cost calculation.

    Variable = sector: total cost variation in percent.
    Variable = sector | origin: exponentiated log contribution of the supplying bloc.
    The exact reconstruction is product(1 + bilateral_percent / 100) - 1.
    These bilateral percentages retain their earlier definition and are not additive.
    """
    frames = []
    for model, res in results.items():
        if "dlnCost" not in res.columns:
            print(f"modèle ({model}) : pas de dlnCost (fichier beta absent), ignoré")
            continue
        long = res.loc[
            res["year"].isin(years),
            ["scenario", "bloc", "sector", "origin", "year", "dlnCost"],
        ].dropna(subset=["dlnCost"]).copy()
        long = add_allsect(add_nonmanuf(long))

        # Sum by origin BEFORE converting log differences into percentage changes.
        total = long.groupby(
            ["scenario", "bloc", "sector", "year"], as_index=False
        )["dlnCost"].sum(min_count=1)
        total["Variable"] = [SECTOR_NAMES.get(s, s) for s in total["sector"]]

        long["Variable"] = [
            IAMC_VARIABLE.format(sector=SECTOR_NAMES.get(s, s), origin=o)
            for s, o in zip(long["sector"], long["origin"])
        ]
        for table in (total, long):
            table["value"] = (np.exp(table["dlnCost"]) - 1) * 100
            table["Case"] = [
                IAMC_SCENARIO.format(model=model, scenario=s) for s in table["scenario"]
            ]
            table["Region (importer)"] = table["bloc"]
            frames.append(table[["Case", "Region (importer)", "Variable", "year", "value"]])

    long = pd.concat(frames, ignore_index=True)
    wide = long.pivot(
        index=["Case", "Region (importer)", "Variable"], columns="year", values="value"
    ).reset_index()
    wide.columns.name = None
    wide.insert(0, "Model", IAMC_MODEL)
    wide.insert(4, "Unit", IAMC_UNIT)
    if simple:
        wide = wide.drop(wide[wide["Variable"].str.contains("|", regex=False, na=False)].index)
    return wide


def export_iamc(results):
    iamc = to_iamc(SIMPLE, results)
    stem = f"ENTICE_T34_{BASE}-{CONTENT}_AllCases_{scenario_name}{BETA_SUFFIX}"
    write_table(iamc, out / f"{stem}.csv", "Final export (IAMC format): change in unit cost (%), Region = importing bloc, Variable = sector total" + ("" if SIMPLE else " or 'sector | supplying bloc'") + ", one column per year.",
                encoding="utf-8-sig", **IAMC_CSV)
    is_bilateral = iamc["Variable"].str.contains("|", regex=False)
    print(
        f"{len(iamc)} lignes : {(~is_bilateral).sum()} totaux sectoriels et "
        f"{is_bilateral.sum()} lignes bilatérales écrites dans {out / stem}.csv"
    )

    return stem

In [ ]:
# ---- VA weighting of the exported cost variations ------------------------------------------------------------------------------------
# Reads the CSV written by the cell above (the comparison with / without VA is shown and exported in the next cell): the aggregates of the CSV are not weighted by the value added. Here the sectors are
# averaged with the STAN VA of the sector IN THE BLOC (VA of the STAN countries of the bloc, summed) as weight, for the manufacturing and for all the sectors.
# The average is made on the log cost variations (dlnCost = log(1 + % / 100), exact: the CSV keeps the full precision), then converted back into %.
# Where there is no VA, there is no weighted value (NaN): blocs without any STAN country (China, RestBRICS, RestAsia, RoW); inside a bloc, sectors without VA
# are left out and the weights are renormalised on the sectors that have one.
VA_FILE = Path(INTERMEDIATE_DIR, "panel_va_STAN.csv")
VA_VAR = "VA_V"                                               # VA_V = current euros | VA_Q = volume
VA_YEAR = YEAR                                                # year of the weights; for a (country, sector) without VA this year, the latest earlier year is used
VA_GROUPS = {                                                 # aggregate: sectors averaged (the aggregates "C" and "H" of AGGREGATES are not components)
    "Manufacturing": lambda s: s.startswith("C") and s not in AGGREGATES,
    "All sectors": lambda s: s not in AGGREGATES and s not in (NONMANUF, ALLSECT),       # NONMANUF and ALLSECT (production-weighted aggregates of the cell above) are not components
}
SEP = " | "                                                   # separator of "sector | origin" in the Variable column (IAMC_VARIABLE)


def load_va_weights(path=VA_FILE, var=VA_VAR, year=VA_YEAR, region_to_block=REGION_TO_BLOCK, default_block=DEFAULT_BLOCK):
    va = pd.read_csv(path, usecols=["Region", "Sector", "annee", var])
    va = va[(va["annee"] <= year) & (va[var] > 0)].dropna(subset=[var])
    last = va.sort_values("annee").groupby(["Region", "Sector"], as_index=False).last()      # latest year <= year, per (country, sector)
    last["bloc"] = last["Region"].map(region_to_block).fillna(default_block)
    weights = last.groupby(["bloc", "Sector"])[var].sum().rename("va").reset_index().rename(columns={"Sector": "sector"})
    return weights, last.groupby("Region")["annee"].agg(["min", "max"])


def average(long, groups, weights=None):
    """Average of the log variations of the sectors of each group, by (case, bloc, origin, year): weights = VA of the sector in the bloc, or simple average
    if weights is None. The result is converted back into %."""
    base = long.dropna(subset=["dln"])
    base = base.assign(va=1.0) if weights is None else base.merge(weights, on=["bloc", "sector"], how="inner")
    keys, res = ["Case", "bloc", "origin", "year"], []
    for label, in_group in groups.items():
        sub = base[base["sector"].map(in_group)].assign(wx=lambda d: d["va"] * d["dln"])
        agg = sub.groupby(keys).agg(wx=("wx", "sum"), w=("va", "sum")).reset_index()
        agg["value"], agg["Group"] = (np.exp(agg["wx"] / agg["w"]) - 1) * 100, label
        res.append(agg[keys + ["Group", "value"]])
    return pd.concat(res, ignore_index=True)


# the CSV of the cell above, back in long format: one row per (case, bloc, sector or "C" aggregate, origin or "Total", year)
def export_va_comparison(stem):
    iamc_csv = pd.read_csv(out / f"{stem}.csv", encoding="utf-8-sig", **IAMC_CSV)
    name_to_code = {name: code for code, name in SECTOR_NAMES.items()}
    long = iamc_csv.melt(id_vars=["Case", "Region (importer)", "Variable"], value_vars=[c for c in iamc_csv.columns if str(c).isdigit()],
                         var_name="year", value_name="value").rename(columns={"Region (importer)": "bloc"})
    part = long["Variable"].str.partition(SEP)
    long["sector"] = part[0].map(name_to_code).fillna(part[0])
    long["origin"] = part[2].replace("", "Total")                  # no origin = total of the sector
    long["year"] = long["year"].astype(int)
    long["dln"] = np.log1p(long["value"] / 100)

    va_weights, va_years = load_va_weights()
    no_va = sorted(set(long["bloc"]) - set(va_weights["bloc"]))
    print(f"VA {VA_VAR}: weights of {va_weights['bloc'].nunique()} blocs ({', '.join(sorted(va_weights['bloc'].unique()))}), "
          f"{va_years.shape[0]} STAN countries, years used {int(va_years['min'].min())}-{int(va_years['max'].max())}")
    print(f"blocs without VA (no VA-weighted value): {no_va}")
    sectors = long.loc[long["origin"] == "Total", "sector"].unique()
    for label, in_group in VA_GROUPS.items():
        in_g = [s for s in sectors if in_group(s)]
        for b in sorted(va_weights["bloc"].unique()):
            miss = [s for s in in_g if s not in set(va_weights.loc[va_weights["bloc"] == b, "sector"])]
            print(f"{label} - {b}: {len(in_g) - len(miss)}/{len(in_g)} sectors with VA" + (f" (without: {', '.join(miss)})" if miss else ""))

    # WITHOUT VA: manufacturing = aggregate "C" of the CSV (computed from the R&D stocks); all sectors = simple average of the sectors (the CSV has no such aggregate)
    without_va = pd.concat([long.loc[long["sector"] == "C", ["Case", "bloc", "origin", "year", "value"]].assign(Group="Manufacturing"),
                            average(long, {"All sectors": VA_GROUPS["All sectors"]})], ignore_index=True).rename(columns={"value": "sans VA"})
    with_va = average(long, VA_GROUPS, va_weights).rename(columns={"value": "avec VA"})
    comparison = without_va.merge(with_va, on=["Case", "bloc", "origin", "year", "Group"], how="left")
    comparison["écart"] = comparison["avec VA"] - comparison["sans VA"]          # in points of % (VA-weighted minus without VA)
    years_all = sorted(comparison["year"].unique())
    export = (comparison.pivot(index=["Case", "bloc", "Group", "origin"], columns="year", values=["sans VA", "avec VA", "écart"])
              .swaplevel(axis=1).reindex(columns=pd.MultiIndex.from_product([years_all, ["sans VA", "avec VA", "écart"]])))
    export.columns = [f"{y} {m}" for y, m in export.columns]
    export.index = export.index.set_names(["Case", "Region (importer)", "Group", "Origin"])
    path_cmp = out / f"{stem}_VA_comparison.csv"
    write_table(export.reset_index().round(4), path_cmp,
                f"Comparison without / with weighting by the value added ({VA_VAR}, STAN, year <= {VA_YEAR}) of the cost variations (%), for Manufacturing and All sectors; 'écart' = avec VA - sans VA, in points of %. "
                f"Empty 'avec VA' = no VA for this bloc (blocs without VA: {', '.join(no_va)}).",
                encoding="utf-8-sig", **IAMC_CSV)
    print(f"{len(export)} lignes (sans / avec pondération par la VA) écrites dans {path_cmp}")

In [ ]:
if NEMESIS_SCENARIO not in ("TariffsAndSub", "Tariff"):
    raise ValueError("Choose TariffsAndSub or Tariff in the settings cell.")
scenarios = [f"{NEMESIS_SCENARIO}_grp{group}" for group in range(1, 6)]
calibrations = ("_low", "_central", "_high")
required = [inputs / f"omega_nemesis_{name}.csv" for name in scenarios]
if "C" in CASES:
    required.extend(inputs / f"prod_nemesis_{name}.csv" for name in scenarios)
required.extend(inputs / f"beta_model1{suffix}.csv" for suffix in calibrations)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing batch inputs: " + ", ".join(missing))

for scenario_name in scenarios:
    OMEGA_FILE = f"omega_nemesis_{scenario_name}.csv"
    PROD_FILE = f"prod_nemesis_{scenario_name}.csv"
    omega_all, prod_all = load_scenario_inputs(OMEGA_FILE, PROD_FILE)
    for BETA_SUFFIX in calibrations:
        print(f"{scenario_name} / {BETA_SUFFIX[1:]}")
        results = calculate_results()
        export_bilateral(results)
        stem = export_iamc(results)
        export_va_comparison(stem)

print(f"Completed {len(scenarios) * len(calibrations)} combinations in {out}")